# Clean-Hour E11: a decision-focused Transformer (PyTorch, GPU)

**Question:** our production model (gradient boosting) predicts each hour on its own. Can a neural network that sees all 76 future hours at once, and is trained to *choose* the cleanest 4-hour window instead of only predicting numbers, pick cleaner hours for a Northern Virginia household or a Loudoun data center?

**Fair test (fixed before scoring 2026):** train 2023 to mid-2025, pick epochs on late 2025, refit on 2023-2025, score once on every complete 2026 day. Same baselines (persistence = tomorrow equals today, hour-of-week average, perfect hindsight) and the same 95% block bootstrap as the main results.

**Kill rule:** WIN only if the decision Transformer beats the production model with an interval below zero at both 24 h and 72 h in at least 3 of 5 grids.

**Before you start:** Runtime > Change runtime type > **T4 GPU**. Then Runtime > **Run all**. The first cell downloads about 1 GB of public grid data (10-20 min).

In [ ]:
!git clone -q -b exp/torch-transformer https://github.com/Harshal623/clean-hour
%cd clean-hour
!pip -q install -r requirements.txt
import torch; print('GPU:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

## 1. Download and cache the public data (EIA-930, Energy-Charts, NESO, Open-Meteo)

In [ ]:
!python -I src/grids.py

## 2. Train both Transformers on 2023-2025 and score them on 2026
Prints each grid as it finishes, then the verdict.

In [ ]:
!python -I -W ignore src/experiments/torch_transformer/run.py

## 3. Results: share of the best possible saving captured (higher is better)

In [ ]:
import pandas as pd, json
S = pd.read_csv('results/experiments/torch_transformer/summary.csv')
for slip in [24, 72]:
    print(f'\n{slip} h of slack, % of the best possible saving')
    print(S[S.slip_h == slip].pivot(index='region', columns='method', values='pct_of_perfect_saving').round(1).to_string())
print('\nTransformer (decision) minus production model, g/kWh, 95% interval (negative = Transformer better):')
t = S[S.method == 'transformer_decision'][['region','slip_h','minus_gbm','minus_gbm_lo','minus_gbm_hi','hourly_mae']]
print(t.round(2).to_string(index=False))
log = json.load(open('results/experiments/torch_transformer/run_log.json'))
print('\nVERDICT:', log['verdict'], '| wins:', log['wins'])
for g in log['grids']: print(g['grid'], 'worst days:', g['worst_days_24h'])

In [ ]:
from IPython.display import Image
Image('results/experiments/torch_transformer/capture_24h.png')

## 4. Save your results
1. Run the next cell and download `torch_results.zip` (send it to the team / Claude so the numbers go into the write-up).
2. **File > Save a copy in GitHub** > repo `Harshal623/clean-hour`, branch `exp/torch-transformer`, path `src/experiments/torch_transformer/colab.ipynb`. That puts this notebook, with outputs, where judges can see it.

In [ ]:
!zip -qr torch_results.zip results/experiments/torch_transformer
from google.colab import files
files.download('torch_results.zip')